# 10-Session OOF Strategy Backtest

Test whether the selected 10-session Elastic Net forecasts survive portfolio construction and transaction costs. Every signal is out of sample for its fold. Because the model and horizon were selected after inspecting these folds, this is economic validation, not a fresh production holdout.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipelines.ml.dataset import get_database_engine, load_adjusted_prices
from pipelines.ml.oof_backtest import run_oof_strategy_backtest
from pipelines.ml.target_study import build_horizon_feature_datasets

In [ ]:
engine = get_database_engine()
dataset = build_horizon_feature_datasets(engine, horizons=(10,))[10]
result = run_oof_strategy_backtest(
    dataset,
    load_adjusted_prices(engine),
    top_k=3,
    min_signals=4,
    transaction_cost_bps=10,
)

print(f"OOF events: {result.decision['oof_event_rows']}")
print(f"Event hit rate: {result.decision['event_hit_rate']:.1%}")

## Forecast Evidence

The historical mean is a legitimate forecast baseline. It is not a rank portfolio baseline because its prediction is identical for every company in a fold.

In [ ]:
display(result.forecast_comparison)
display(result.event_diagnostics)

## Cost-Aware Portfolio Paths

Positions enter at the event feature-session close, earn returns from the next session onward, and expire at the 10-session target date. Net paths deduct 10 bps per unit of one-way turnover.

In [ ]:
display(result.performance)

daily = result.daily_returns.set_index('trading_date')
paths = (1 + daily[[
    'long_short_gross_return',
    'long_short_net_return',
    'long_only_net_return',
    'spy_return',
]]).cumprod()
paths.columns = ['Long-short gross', 'Long-short net', 'Long-only net', 'SPY']

ax = paths.plot(figsize=(12, 6), linewidth=2)
ax.set_title('Out-of-fold strategy wealth, starting at 1.0')
ax.set_xlabel('Trading date')
ax.set_ylabel('Growth of $1')
ax.grid(alpha=0.25)
plt.show()

## Fold Stability

A pooled return can hide regime dependence. Inspect net return, Sharpe, drawdown, invested days, and turnover in each original validation window.

In [ ]:
display(result.fold_performance)

fold_returns = result.fold_performance.pivot(
    index='fold',
    columns='strategy',
    values='total_return',
)
ax = fold_returns.plot.bar(figsize=(12, 5))
ax.set_title('Total return by untouched validation fold')
ax.set_xlabel('Fold')
ax.set_ylabel('Total return')
ax.axhline(0, color='black', linewidth=0.8)
ax.grid(axis='y', alpha=0.25)
plt.show()

## Governance Decision

The result remains research-only regardless of backtest performance. Promotion requires observations that arrived after the target-study decision.

In [ ]:
result.decision